# Green5G no Google Colab

Este notebook roda o projeto **Green5G** e mostra os resultados em gráficos.

O Green5G é uma IA que decide quando uma célula 5G de capacidade pode entrar em **sleep** para economizar energia, sem prejudicar os usuários.

**Como usar**
1. Menu **Ambiente de execução → Executar tudo**.
2. A primeira célula baixa o projeto do GitHub sozinha. Se não conseguir, ela pede o `Green5G.zip`: clique em **Escolher arquivos** e envie.
3. Para testar outros cenários, mude os valores da seção 2 e rode de novo a partir dela.

## 1. Carregar o projeto

In [ ]:
import os, sys, subprocess, zipfile

REPO = "https://github.com/ClethieB-cmd/Green5G.git"

if os.path.basename(os.getcwd()) != "Green5G":
    if not os.path.isdir("Green5G"):
        # Baixa o projeto do GitHub. Se não der certo, pede o Green5G.zip.
        if subprocess.run(["git", "clone", "-q", REPO]).returncode != 0:
            from google.colab import files
            enviado = files.upload()
            zipfile.ZipFile(next(iter(enviado))).extractall(".")
    os.chdir("Green5G")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Projeto carregado em", os.getcwd())

## 2. Parâmetros

Mude aqui e rode as células seguintes de novo.

`ARQUIVO_CSV = None` usa dados simulados. Para dados reais, envie o CSV pelo ícone de pasta, na barra à esquerda, e escreva o nome do arquivo. As colunas precisam ser `timestamp, celula_id, demanda` e, opcionalmente, `solar_w`.

In [ ]:
ARQUIVO_CSV = None        # ex.: "dados_operadora.csv"
CAPACIDADE_MACRO = 45     # usuários extras que a macrocélula aguenta sem degradar (SUPOSIÇÃO: confirmar com a engenharia)
MARGEM_SLEEP = 0.85       # a célula dorme se o pico P90 previsto < MARGEM_SLEEP x CAPACIDADE_MACRO
CELULA = "CEL-01"         # célula do gráfico detalhado
DIAS_NO_GRAFICO = 3

## 3. Dados

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from green5g.config import Config
from green5g.dados import simular_rede, carregar_csv, regularizar, eh_fim_de_semana
from green5g.avaliacao import backtest, relatorio_markdown
from green5g import dashboard as dash

cfg = Config()
cfg.politica.capacidade_absorcao_macro = CAPACIDADE_MACRO
cfg.politica.margem_sleep = MARGEM_SLEEP

if ARQUIVO_CSV:
    df, freq, qualidade = carregar_csv(ARQUIVO_CSV)
else:
    df, freq, qualidade = regularizar(
        simular_rede(dias=70, freq_min=15, n_celulas=6, feriados=cfg.geral.feriados))

print(f"{qualidade['_geral']['celulas']} células | medição a cada {freq} min | "
      f"{df['timestamp'].min():%d/%m/%Y} a {df['timestamp'].max():%d/%m/%Y}")
display(df.head())

### Como é a demanda de cada célula

Este gráfico mostra a média por hora do dia, separando dias úteis de fins de semana e feriados. Cada célula tem um padrão próprio, e a IA aprende isso sozinha, sem horários fixos no código.

In [ ]:
celulas = sorted(df["celula_id"].unique())[:6]
fig, eixos = plt.subplots(2, 3, figsize=(16, 7), sharey=True, facecolor=dash.COR["fundo"])
for ax, cel in zip(eixos.flat, celulas):
    g = df[df["celula_id"] == cel].copy()
    g["hora"] = g["timestamp"].dt.hour
    g["fds"] = eh_fim_de_semana(g["timestamp"], cfg.geral.feriados)
    perfil = g.groupby(["fds", "hora"])["demanda"].mean().unstack(0)
    ax.plot(perfil.index, perfil[False], color=dash.COR["azul"], linewidth=2, label="Dia útil")
    if True in perfil:
        ax.plot(perfil.index, perfil[True], color=dash.COR["laranja"], linewidth=2, label="Fim de semana / feriado")
    ax.axhline(cfg.politica.capacidade_absorcao_macro, color=dash.COR["texto2"], linestyle="--", linewidth=1)
    ax.set_xticks(range(0, 24, 4))
    dash.estilo(ax, cel)
for ax in eixos.flat[len(celulas):]:
    ax.axis("off")
eixos.flat[0].legend(frameon=False, fontsize=9)
eixos.flat[0].set_ylabel("Usuários ativos (média)")
fig.suptitle("Perfil médio de demanda por célula (linha tracejada = capacidade da macro)",
             x=0.02, ha="left", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## 4. Avaliação (backtest walk-forward)

O teste simula o uso real. Para cada uma das últimas 3 semanas, o modelo é treinado **só com o passado** e decide de 15 em 15 minutos, sem ver o futuro.

Quatro políticas são comparadas:
- **Sempre ativa**: a referência.
- **Regras v1**: o código original.
- **Green5G IA**: o modelo novo.
- **Oráculo**: um limite teórico que conhece o futuro.

Leva cerca de 15 segundos.

In [ ]:
res = backtest(df, freq, cfg)

colunas = {
    "energia_rede_kwh": "Energia da rede (kWh)",
    "economia_pct": "Economia (%)",
    "tempo_sleep_pct": "Tempo em sleep (%)",
    "violacoes_qos_pct_tempo": "Violações de QoS (% do tempo)",
    "usuarios_hora_afetados": "Usuários·h afetados",
    "religamentos_por_celula_dia": "Religamentos/célula/dia",
}
tabela = pd.DataFrame(res["politicas"]).T[list(colunas)].rename(columns=colunas).astype(float).round(2)
tabela["Economia (%)"] = tabela["Economia (%)"].astype(object).where(tabela["Economia (%)"].notna(), "referência")
display(tabela)

pg = res["previsao_geral"]
print(f"Previsão do pico da próxima hora: erro médio {pg['mae_p50']:.1f} usuários | "
      f"cobertura do P90 {pg['cobertura_p90_pct']:.1f}% (meta 90%) | "
      f"{pg['melhoria_vs_ingenuo_pct']:.0f}% melhor que repetir a semana passada")

## 5. Gráficos

### Uma célula em operação

- **Linha azul**: demanda real.
- **Faixa azul-clara**: previsão do pico da próxima hora, do valor provável (P50) ao pessimista (P90).
- **Faixas verdes**: momentos em que a IA colocou a célula em sleep.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5.5), facecolor=dash.COR["fundo"])
dash.grafico_celula(ax, res, cfg, celula=CELULA, dias=DIAS_NO_GRAFICO)
plt.show()

### Quanto cada política gasta, e o risco de cada uma

Na direita, cada ponto azul é a IA com uma margem de sleep diferente. Quanto mais para cima, mais economia; quanto mais para a direita, mais usuários afetados. O ideal é ficar **no alto e à esquerda**.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 5.5), facecolor=dash.COR["fundo"])
dash.grafico_consumo(ax1, res)
dash.grafico_risco(ax2, res, cfg)
fig.tight_layout()
plt.show()

### Qualidade da previsão e quando a célula dorme

- **Esquerda**: erro médio da IA por hora do dia, comparado com simplesmente repetir o que aconteceu na semana passada.
- **Direita**: em que horas e dias a IA coloca as células em sleep.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 5.5), facecolor=dash.COR["fundo"])
dash.grafico_erro_hora(ax1, res)
dash.grafico_mapa_sleep(ax2, res)
fig.tight_layout()
plt.show()

## 6. Por que a IA decidiu isso?

Cada mudança de estado vem com um motivo legível. É isso que a operadora usa para auditar o sistema.

In [ ]:
r = res["_resultados"]["Green5G IA"]
r = r[r["celula_id"] == CELULA]
mudancas = r[r["decisao"] != r["modo"]].rename(columns={
    "timestamp": "momento da decisão", "modo": "estava", "decisao": "passa para",
    "demanda": "demanda agora", "p50": "pico previsto P50", "p90": "pico previsto P90"})
display(mudancas[["momento da decisão", "estava", "passa para", "demanda agora", "pico previsto P50",
                  "pico previsto P90", "motivo"]].head(15).round({"demanda agora": 1, "pico previsto P50": 1, "pico previsto P90": 1}))

## 7. Relatório completo

In [ ]:
display(Markdown(relatorio_markdown(res, cfg, qualidade, freq)))

## 8. Painel web (front-end)

Gera o painel interativo, mostra aqui embaixo e baixa o arquivo `painel.html`. Depois, é só abrir esse arquivo em qualquer navegador.

In [ ]:
from IPython.display import HTML
from green5g.painel import gerar_painel

perfis = (df.drop_duplicates("celula_id").set_index("celula_id")["perfil"].to_dict()
          if "perfil" in df.columns else None)
gerar_painel(res, cfg, freq, "painel.html", perfis=perfis, simulado=not ARQUIVO_CSV)
display(HTML(open("painel.html", encoding="utf-8").read()))

try:
    from google.colab import files
    files.download("painel.html")
except ImportError:
    print("Painel salvo em", os.path.abspath("painel.html"))

---
**Lembrete:** com dados simulados, estes números mostram que o método funciona, não quanto uma operadora real vai economizar.

O resultado que vale é o obtido com `ARQUIVO_CSV` apontando para os dados reais, depois de ajustar os parâmetros marcados como SUPOSIÇÃO em `green5g/config.py`.